# BankAssist Triage - BANKING77 Dataset Exploration Tasks

This notebook is for **you** to complete the dataset exploration and preprocessing work step by step.

It intentionally contains **no solutions, no working code, and no completed outputs**.

Use this notebook to investigate BANKING77, document decisions, and prepare the dataset before instruction formatting.


## Rules For This Notebook

- Do not start tokenization in this notebook.
- Do not start model loading or LoRA fine-tuning in this notebook.
- Do not convert to instruction-following conversations until the enriched dataset has been verified.
- Prefer Hugging Face Dataset APIs over pandas unless there is a clear reason.
- Preserve the original integer `label` column throughout preprocessing.
- Use BANKING77 `ClassLabel` metadata to convert label IDs into intent names.
- Record important findings in `research/dataset_notes.md` after you verify them.


## Current Business Scope

Version 1 uses these nine BANKING77 intents:

1. `lost_or_stolen_card`
2. `compromised_card`
3. `card_payment_not_recognised`
4. `cash_withdrawal_not_recognised`
5. `wrong_amount_of_cash_received`
6. `transfer_not_received_by_recipient`
7. `declined_card_payment`
8. `card_not_working`
9. `change_pin`

Deferred for a later dataset version:

- `refund_not_showing_up`
- `out_of_scope`


# Master Task Tracker

Update the status column as you complete each task.

| ID | Task | Status | Expected Output |
|---|---|---|---|
| D01 | Set up notebook imports and project path | To do | Required libraries imported without errors |
| D02 | Load BANKING77 | To do | Raw Hugging Face `DatasetDict` loaded |
| D03 | Inspect dataset splits | To do | Split names and row counts recorded |
| D04 | Inspect raw schema | To do | Column names and feature types recorded |
| D05 | Inspect one raw record | To do | Example raw record reviewed |
| D06 | Inspect label metadata | To do | All intent label names available from `ClassLabel` |
| D07 | Confirm selected intents exist | To do | All nine selected intents verified in metadata |
| D08 | Convert selected intent names to label IDs | To do | Mapping from selected intent name to original label ID |
| D09 | Filter dataset to selected intents | To do | Filtered train/test dataset containing only selected intents |
| D10 | Verify filtered dataset schema | To do | `text` and original `label` still present |
| D11 | Define business rules | To do | Rule mapping for all nine selected intents |
| D12 | Validate business-rule coverage | To do | Every selected intent has exactly one rule |
| D13 | Enrich records | To do | Added `intent`, `priority`, `route`, `requires_human_review` |
| D14 | Verify enriched schema | To do | Required enriched columns present, original `label` preserved |
| D15 | Inspect enriched examples | To do | Several records manually reviewed |
| D16 | Compute class distribution | To do | Train/test counts per selected intent |
| D17 | Check missing values | To do | Missing/empty values counted per split |
| D18 | Check duplicate messages | To do | Duplicate count per split |
| D19 | Check train/test leakage | To do | Exact text overlap between train and test checked |
| D20 | Analyze message lengths | To do | Min/mean/p50/p95/max length summary |
| D21 | Review examples per intent | To do | Sample examples inspected for every selected intent |
| D22 | Review ambiguous intent groups | To do | Confusing examples documented |
| D23 | Decide cleaning policy | To do | Keep/remove rules documented |
| D24 | Decide validation split strategy | To do | Validation percentage, stratification, and seed selected |
| D25 | Create validation split | To do | Train/validation/test split plan or artifact prepared |
| D26 | Save enriched dataset | To do | Reproducible saved dataset artifact |
| D27 | Update dataset notes | To do | Findings written to `research/dataset_notes.md` |
| D28 | Handoff to instruction formatting | To do | Clear next-step notes for System/User/Assistant conversion |


# Phase 1 - Raw Dataset Inspection

Complete these tasks before filtering anything.


## Task D01 - Set Up Notebook Imports And Project Path

Goal:

Prepare the notebook environment so you can load datasets and reuse project code when needed.

Acceptance criteria:

- The notebook can import the required libraries.
- The notebook can access project files under `src/` if needed.
- No dataset transformation happens in this task.

Notes to record:

- Which Python environment/kernel is being used.
- Whether `datasets` is installed and importable.


In [130]:
from pathlib import Path
from collections import Counter
import json
import sys
import platform

try:
    import datasets
    from datasets import load_dataset
except ImportError as exc:
    raise ImportError("'datasets' is not installed in the current notebook environment") from exc

PROJECT_ROOT = Path.cwd().parent
SRC_DIR = PROJECT_ROOT / "src"

if not SRC_DIR.is_dir():
    raise NotADirectoryError(f"Source directory does not exist: { SRC_DIR}")

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))


print(f"Python executable: {sys.executable}")
print(f"Python version: {platform.python_version()}")
print(f"datasets version: {datasets.__version__}")
print(f"Source directory: {SRC_DIR}")
print(f"Source directory accessible: {SRC_DIR.is_dir()}")


Python executable: /opt/anaconda3/envs/bankassist/bin/python
Python version: 3.11.15
datasets version: 5.0.0
Source directory: /Users/usmanumar/Desktop/Projects/bankassist-triage/src
Source directory accessible: True


## Task D02 - Load BANKING77

Goal:

Load the raw `PolyAI/banking77` dataset from Hugging Face.

Acceptance criteria:

- The raw dataset object is available in the notebook.
- The dataset has not been filtered or modified yet.

Notes to record:

- Dataset name.
- Dataset revision, if one is used.
- Any loading warnings or errors.


In [131]:
DATASET_NAME = 'PolyAI/banking77'
DATASET_REVISION = 'main'

try:
    dataset = load_dataset(DATASET_NAME, revision=DATASET_REVISION)
except Exception as exc:
    raise RuntimeError(f'Dataset can not be loaded: {exc}') from exc

print(f'Dataset Name: {DATASET_NAME}')
print(f'Dataset Revision: {DATASET_REVISION}')


Using the latest cached version of the dataset since PolyAI/banking77 couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'default' at /Users/usmanumar/.cache/huggingface/datasets/PolyAI___banking77/default/0.0.0/796a4623935746f71378f0ebd435635a8ce08e50 (last modified on Tue Jul 21 17:56:03 2026).


Dataset Name: PolyAI/banking77
Dataset Revision: main


## Task D03 - Inspect Dataset Splits

Goal:

Understand what official splits BANKING77 provides.

Acceptance criteria:

- Split names are listed.
- Row count for each split is recorded.

Questions to answer:

- Does the dataset provide an official test split?
- Should the official test split be preserved for final evaluation?


In [132]:
keys = list(dataset.keys())
TRAIN_SPLIT = keys[0]
TEST_SPLIT = keys[1]

TRAIN_ROWS = dataset['train'].num_rows
TEST_ROWS = dataset['train'].num_rows

print("Splits and Row numbers are recorded\n", dataset)
print('Split Names:', TRAIN_SPLIT, TEST_SPLIT)
print('Row Numbers:', TRAIN_ROWS, TEST_ROWS)
print(f'Dataset has an official split of train and test')
print(f'Yes, test split should be preserved for final evaluation')

Splits and Row numbers are recorded
 DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 10003
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 3080
    })
})
Split Names: train test
Row Numbers: 10003 10003
Dataset has an official split of train and test
Yes, test split should be preserved for final evaluation


## Task D04 - Inspect Raw Schema

Goal:

Confirm the columns and feature types in the raw dataset.

Acceptance criteria:

- Column names are recorded.
- Feature types are recorded.
- The type of the `label` column is understood.

Questions to answer:

- Is `text` stored as a string?
- Is `label` stored as an integer class label?
- Where are readable label names stored?


In [133]:
TEXT_FEATURE = dataset["train"].features["text"]
LABEL_FEATURE = dataset["train"].features["label"]

sample_label = dataset["train"][3]["label"]

print(f"Column names: {dataset.column_names}")
print(f"Text feature: {TEXT_FEATURE}")
print(f"Text dtype: {TEXT_FEATURE.dtype}")

print(f"Label feature: {LABEL_FEATURE}")
print(f"Label feature type: {type(LABEL_FEATURE)}")
print(f"Sample label: {sample_label}")
print(f"Sample label Python type: {type(sample_label)}")

if hasattr(LABEL_FEATURE, "names"):
    print(f"Readable label names: {LABEL_FEATURE.names}")
    print(f"Sample readable label: {LABEL_FEATURE.int2str(sample_label)}")

Column names: {'train': ['text', 'label'], 'test': ['text', 'label']}
Text feature: Value('string')
Text dtype: string
Label feature: ClassLabel(names=['activate_my_card', 'age_limit', 'apple_pay_or_google_pay', 'atm_support', 'automatic_top_up', 'balance_not_updated_after_bank_transfer', 'balance_not_updated_after_cheque_or_cash_deposit', 'beneficiary_not_allowed', 'cancel_transfer', 'card_about_to_expire', 'card_acceptance', 'card_arrival', 'card_delivery_estimate', 'card_linking', 'card_not_working', 'card_payment_fee_charged', 'card_payment_not_recognised', 'card_payment_wrong_exchange_rate', 'card_swallowed', 'cash_withdrawal_charge', 'cash_withdrawal_not_recognised', 'change_pin', 'compromised_card', 'contactless_not_working', 'country_support', 'declined_card_payment', 'declined_cash_withdrawal', 'declined_transfer', 'direct_debit_payment_not_recognised', 'disposable_card_limits', 'edit_personal_details', 'exchange_charge', 'exchange_rate', 'exchange_via_app', 'extra_charge_on_s

## Task D05 - Inspect One Raw Record

Goal:

Look at a small number of raw examples before any transformation.

Acceptance criteria:

- At least one raw training example is displayed.
- At least one raw test example is displayed.
- You understand what the original records look like.

Questions to answer:

- Does each record contain only `text` and `label`?
- Is any business information already present?


In [134]:
RAW_TRAIN = dataset['train'][0]
RAW_TEST = dataset['test'][0]

print(RAW_TRAIN)
print(RAW_TEST)

print('Yes, only text and label')
print('Yes. Business context is present in the customer-query text, which relates to card-support issues. No separate business metadata is visible in these examples.”')

{'text': 'I am still waiting on my card?', 'label': 11}
{'text': 'How do I locate my card?', 'label': 11}
Yes, only text and label
Yes. Business context is present in the customer-query text, which relates to card-support issues. No separate business metadata is visible in these examples.”


# Phase 2 - Label Metadata And Intent Selection

Complete these tasks before filtering the dataset.


## Task D06 - Inspect Label Metadata

Goal:

Use BANKING77 `ClassLabel` metadata to retrieve readable intent names.

Acceptance criteria:

- Total number of original intent classes is recorded.
- All readable intent names can be accessed.
- The mapping between label ID and label name is understood.

Questions to answer:

- How many original intents are in BANKING77?
- Are labels represented as integers in records?
- How will you convert a label integer into a readable intent name?


In [135]:
TOTAL_NUMBER_OF_CLASSES = dataset['train'].features['label']
print(TOTAL_NUMBER_OF_CLASSES.num_classes)

READABLE_INTENT_NAMES = dataset['train'].features['label']
print(READABLE_INTENT_NAMES)

for id, intent in enumerate(READABLE_INTENT_NAMES.names):
    print(f'ID: {id} - Intent: {intent}')
print('Intent is mapped to the Intent ID of its position in the array.')

print(f'Original Intents:  {TOTAL_NUMBER_OF_CLASSES}')

dataset['train'][0]
print('Labels are represented as integers')

feature_label = dataset['train'].features['label']
id_label = dataset['train'][200]['label']
print('Convert id -> intent: ', feature_label.int2str(id_label))



77
ClassLabel(names=['activate_my_card', 'age_limit', 'apple_pay_or_google_pay', 'atm_support', 'automatic_top_up', 'balance_not_updated_after_bank_transfer', 'balance_not_updated_after_cheque_or_cash_deposit', 'beneficiary_not_allowed', 'cancel_transfer', 'card_about_to_expire', 'card_acceptance', 'card_arrival', 'card_delivery_estimate', 'card_linking', 'card_not_working', 'card_payment_fee_charged', 'card_payment_not_recognised', 'card_payment_wrong_exchange_rate', 'card_swallowed', 'cash_withdrawal_charge', 'cash_withdrawal_not_recognised', 'change_pin', 'compromised_card', 'contactless_not_working', 'country_support', 'declined_card_payment', 'declined_cash_withdrawal', 'declined_transfer', 'direct_debit_payment_not_recognised', 'disposable_card_limits', 'edit_personal_details', 'exchange_charge', 'exchange_rate', 'exchange_via_app', 'extra_charge_on_statement', 'failed_transfer', 'fiat_currency_support', 'get_disposable_virtual_card', 'get_physical_card', 'getting_spare_card', 'g

## Task D07 - Confirm Selected Intents Exist

Goal:

Verify that all nine Version 1 selected intents exist in BANKING77 metadata.

Acceptance criteria:

- Every selected intent is found in the metadata.
- Any spelling or capitalization issues are identified.

Questions to answer:

- Do all selected intents exist exactly as written?
- Are any similar labels present that are not part of Version 1?


In [154]:
from spellchecker import SpellChecker
import re

spell = SpellChecker()
spell.word_frequency.load_words(["app", 'atm', 'dont',"would've", "where'd", ])

featured_intent = dataset['train'].features['label']

selected_intents = [
    "lost_or_stolen_card",
    "compromised_card",
    "card_payment_not_recognised",
    "cash_withdrawal_not_recognised",
    "wrong_amount_of_cash_received",
    "transfer_not_received_by_recipient",
    "declined_card_payment",
    "card_not_working",
    "change_pin",
]

missing_intents = []

for intent in selected_intents:
    if intent not in featured_intent.names:
        missing_intents.append(intent)

if missing_intents:
    print("The following intents do not exist in the metadata:")

    for intent in missing_intents:
        print(intent)
else:
    print("All selected intents are available in the metadata.")


All selected intents are available in the metadata.


In [151]:
label_feature = dataset['train'].features['label']

selected_label_ids = {
    label_feature.str2int(intent)
    for intent in selected_intents
}

featured_data = dataset['train'].filter(
    lambda row: row['label'] in selected_label_ids
)


def correct_text(text):
    def replace_word(match):
        word = match.group(0)
        corrected = spell.correction(word.lower()) or word

        if word.istitle():
            corrected = corrected.title()
        elif word.isupper():
            corrected = corrected.upper()

        return corrected

    return re.sub(r"\b[A-Za-z']+\b", replace_word, text)


for text in featured_data["text"]:
    corrected_text = correct_text(text)

    print("Original: ", text)
    print("Corrected:", corrected_text)
    print("-" * 50)

Original:  I can't use my card because it is not working.
Corrected: I can't use my card because it is not working.
--------------------------------------------------
Original:  I can't seem to be able to use my card
Corrected: I can't seem to be able to use my card
--------------------------------------------------
Original:  My card isn't working at all, I need assistance. It's really frustrating.
Corrected: My card isn't working at all, I need assistance. it's really frustrating.
--------------------------------------------------
Original:  Can you tell me what the problem with my card is? It was declined at a restaurant today.
Corrected: Can you tell me what the problem with my card is? It was declined at a restaurant today.
--------------------------------------------------
Original:  How do I fix a broken card?
Corrected: How do I fix a broken card?
--------------------------------------------------
Original:  My card was working but now it doesn't.
Corrected: My card was working

## Task D08 - Convert Selected Intent Names To Label IDs

Goal:

Create a reliable mapping from selected intent names to original BANKING77 label IDs.

Acceptance criteria:

- Each selected intent has one label ID.
- The label IDs come from `ClassLabel` metadata.
- No label ID is manually guessed.

Expected output:

A table or dictionary showing:

- intent name
- original BANKING77 label ID


In [175]:
selected_intents = [
    "lost_or_stolen_card",
    "compromised_card",
    "card_payment_not_recognised",
    "cash_withdrawal_not_recognised",
    "wrong_amount_of_cash_received",
    "transfer_not_received_by_recipient",
    "declined_card_payment",
    "card_not_working",
    "change_pin",
]

selected_label = dataset['train'].features['label']

intent_to_label_id = {
    intent: selected_label.str2int(intent)
    for intent in selected_intents
}

print(intent_to_label_id)

{'lost_or_stolen_card': 41, 'compromised_card': 22, 'card_payment_not_recognised': 16, 'cash_withdrawal_not_recognised': 20, 'wrong_amount_of_cash_received': 75, 'transfer_not_received_by_recipient': 66, 'declined_card_payment': 25, 'card_not_working': 14, 'change_pin': 21}


In [178]:
from datasets import ClassLabel

label_feature = dataset["train"].features["label"]

print(label_feature)
print(type(label_feature))
print(isinstance(label_feature, ClassLabel))

ClassLabel(names=['activate_my_card', 'age_limit', 'apple_pay_or_google_pay', 'atm_support', 'automatic_top_up', 'balance_not_updated_after_bank_transfer', 'balance_not_updated_after_cheque_or_cash_deposit', 'beneficiary_not_allowed', 'cancel_transfer', 'card_about_to_expire', 'card_acceptance', 'card_arrival', 'card_delivery_estimate', 'card_linking', 'card_not_working', 'card_payment_fee_charged', 'card_payment_not_recognised', 'card_payment_wrong_exchange_rate', 'card_swallowed', 'cash_withdrawal_charge', 'cash_withdrawal_not_recognised', 'change_pin', 'compromised_card', 'contactless_not_working', 'country_support', 'declined_card_payment', 'declined_cash_withdrawal', 'declined_transfer', 'direct_debit_payment_not_recognised', 'disposable_card_limits', 'edit_personal_details', 'exchange_charge', 'exchange_rate', 'exchange_via_app', 'extra_charge_on_statement', 'failed_transfer', 'fiat_currency_support', 'get_disposable_virtual_card', 'get_physical_card', 'getting_spare_card', 'gett

# Phase 3 - Filtering

Complete these tasks before adding business fields.


## Task D09 - Filter Dataset To Selected Intents

Goal:

Create a filtered dataset containing only the nine selected intents.

Acceptance criteria:

- Both train and test splits are filtered.
- Only selected label IDs remain.
- The original integer `label` column is preserved.

Questions to answer:

- How many examples remain in train?
- How many examples remain in test?


## Task D10 - Verify Filtered Dataset Schema

Goal:

Confirm filtering did not remove required fields.

Acceptance criteria:

- `text` remains present.
- `label` remains present.
- `label` values are still original BANKING77 integer IDs.

Questions to answer:

- Did filtering change the label IDs?
- Should label IDs be remapped at this stage?


# Phase 4 - Business Rule Enrichment

Complete these tasks after filtering.


## Task D11 - Define Business Rules

Goal:

Create the rule mapping from intent to business decision fields.

Each selected intent needs:

- `priority`
- `route`
- `requires_human_review`

Acceptance criteria:

- All nine selected intents have rules.
- No unsupported intent appears in the business rules.
- Boolean values are real booleans, not strings.


## Task D12 - Validate Business-Rule Coverage

Goal:

Check that selected intents and business rules match exactly.

Acceptance criteria:

- No selected intent is missing a rule.
- No extra business-rule intent exists outside the selected scope.
- Any mismatch fails loudly.

Questions to answer:

- What should happen if a selected intent has no rule?
- What should happen if a rule exists for an unsupported intent?


## Task D13 - Enrich Records

Goal:

Add business fields to every filtered record.

Expected output record shape:

```json
{
  "text": "...",
  "label": 14,
  "intent": "...",
  "priority": "...",
  "route": "...",
  "requires_human_review": true
}
```

Acceptance criteria:

- `intent` is derived from the original label ID.
- Business fields are attached from the rule mapping.
- The original `label` column is not removed.


## Task D14 - Verify Enriched Schema

Goal:

Confirm the enriched dataset has the exact fields required for the next stage.

Acceptance criteria:

- `text` exists.
- `label` exists.
- `intent` exists.
- `priority` exists.
- `route` exists.
- `requires_human_review` exists.

Questions to answer:

- Are field names consistent across train and test?
- Are booleans stored as booleans?


## Task D15 - Inspect Enriched Examples

Goal:

Manually inspect enriched examples to catch obvious mapping mistakes.

Acceptance criteria:

- At least several records are reviewed.
- The intent matches the original message.
- The priority, route, and human-review fields match the business rules.

Questions to answer:

- Do high-risk messages route correctly?
- Do lower-risk self-service messages avoid unnecessary human review?


# Phase 5 - Dataset Quality Checks

Complete these tasks before creating the final training/validation split.


## Task D16 - Compute Class Distribution

Goal:

Understand class balance after filtering.

Acceptance criteria:

- Count examples per selected intent in train.
- Count examples per selected intent in test.
- Identify largest and smallest classes.

Questions to answer:

- Is the train split balanced?
- Is the test split balanced?
- Should macro-F1 be used later?


## Task D17 - Check Missing Values

Goal:

Find missing or empty values before instruction formatting.

Acceptance criteria:

- Empty or whitespace-only messages are counted.
- Missing `label` values are counted.
- Missing enriched business fields are counted.

Decision to document:

- Keep, remove, or fix affected records.


## Task D18 - Check Duplicate Messages

Goal:

Identify exact duplicate messages within each split.

Acceptance criteria:

- Duplicate count for train is recorded.
- Duplicate count for test is recorded.
- Example duplicates are inspected if any exist.

Decision to document:

- Whether duplicates should be kept or removed.


## Task D19 - Check Train/Test Leakage

Goal:

Ensure exact same messages do not appear in both train and test.

Acceptance criteria:

- Exact text overlap between train and test is counted.
- Overlapping examples are listed if any exist.

Decision to document:

- Whether overlapping records must be removed from train, test, or both.


## Task D20 - Analyze Message Lengths

Goal:

Understand message length before later tokenizer work.

Acceptance criteria:

- Minimum length recorded.
- Mean length recorded.
- Median or p50 length recorded.
- p95 length recorded.
- Maximum length recorded.

Important note:

At this stage, word count is enough. Model-token count should be checked later with the Qwen tokenizer.


## Task D21 - Review Examples Per Intent

Goal:

Manually inspect examples from every selected intent.

Acceptance criteria:

- Several examples reviewed for each of the nine intents.
- Any suspicious examples are recorded.
- Any examples that reveal ambiguity are recorded.

Questions to answer:

- Do examples clearly match their assigned intent?
- Are some classes easier because of obvious keywords?
- Are some messages realistic support-ticket messages?


## Task D22 - Review Ambiguous Intent Groups

Goal:

Identify examples that may confuse the model.

Groups to review:

| Group | Intents |
|---|---|
| Card security | `lost_or_stolen_card`, `compromised_card` |
| Fraud transactions | `card_payment_not_recognised`, `cash_withdrawal_not_recognised` |
| Card support | `declined_card_payment`, `card_not_working` |
| Money movement | `transfer_not_received_by_recipient`, `wrong_amount_of_cash_received` |

Acceptance criteria:

- Ambiguous examples are documented.
- The reason for ambiguity is written down.
- You decide whether each example should be kept, removed, or simply documented.


## Task D23 - Decide Cleaning Policy

Goal:

Define what, if anything, should be cleaned before instruction formatting.

Acceptance criteria:

- Missing-value policy documented.
- Duplicate policy documented.
- Leakage policy documented.
- Ambiguous-example policy documented.

Questions to answer:

- Are any examples removed?
- If records are removed, is the removal reproducible?
- Does cleaning preserve the official test split?


# Phase 6 - Split Planning

Complete these tasks before the instruction-formatting notebook or script.


## Task D24 - Decide Validation Split Strategy

Goal:

Choose how to create validation data for training.

Recommended decisions to evaluate:

- Use only the official train split for train/validation.
- Preserve the official test split for final evaluation.
- Use a stratified split by `intent`.
- Choose and record a fixed random seed.

Acceptance criteria:

- Validation percentage selected.
- Random seed selected.
- Stratification decision documented.
- Test split preservation decision documented.


## Task D25 - Create Validation Split

Goal:

Create train and validation splits from the enriched official training data.

Acceptance criteria:

- Training split created.
- Validation split created.
- Official test split preserved.
- Class counts checked after splitting.
- Split settings are reproducible.

Expected artifact:

A saved dataset version or clear handoff notes for the next preprocessing script.


# Phase 7 - Save And Document

Complete these tasks before leaving this notebook.


## Task D26 - Save Enriched Dataset

Goal:

Save the enriched dataset in a reproducible format.

Acceptance criteria:

- Dataset saved under `data/processed/` or another documented path.
- The saved artifact can be loaded again.
- The saved artifact contains expected columns.

Questions to answer:

- What is the dataset version name?
- What exact command or code reproduces it?


## Task D27 - Update Dataset Notes

Goal:

Move verified findings from this notebook into the research documentation.

Update:

- `research/dataset_notes.md`

Acceptance criteria:

- Selected intents documented.
- Class counts documented.
- Missing-value results documented.
- Duplicate/leakage results documented.
- Cleaning decisions documented.
- Final dataset artifact path documented.


## Task D28 - Handoff To Instruction Formatting

Goal:

Prepare the next step without doing it in this notebook.

Acceptance criteria:

- Confirm enriched dataset is ready.
- Confirm validation split decision is documented.
- Confirm tokenization has not started.
- Write a short note describing the next required transformation.

Next step after this notebook:

Convert each enriched record into a System/User/Assistant instruction-following example where the assistant output is valid JSON only.
